In [1]:
import os
import sys
os.chdir("..")
sys.path.append(os.getcwd())
import pandas as pd
import duckdb
from pathlib import Path
from config import RESEARCH_ROOT, NSE_DB_PATH

In [2]:
RESEARCH_DB_PATH = RESEARCH_ROOT / "research.db"
con = duckdb.connect(RESEARCH_DB_PATH)
con.execute(f"ATTACH '{NSE_DB_PATH}' AS nse (READ_ONLY)") 

In [3]:
print(f"Connected to DuckDB: {RESEARCH_DB_PATH.resolve()}")

Connected to DuckDB: E:\Projects\embR\data\research.db


In [4]:
# read from nse, write to research.db
con.execute("""
    CREATE OR REPLACE TABLE forward_returns AS
    WITH daily AS (
        SELECT
            trade_date,
            index_name,
            close
        FROM nse.market_activity_index
        WHERE index_name = 'Nifty 50'   -- adjust to exact name in your data
        ORDER BY trade_date
    )
    SELECT
        d.trade_date,
        d.index_name,
        d.close,

        -- forward returns
        ROUND((f1.close - d.close) / d.close * 100, 4) AS fwd_ret_1d,
        ROUND((f5.close - d.close) / d.close * 100, 4) AS fwd_ret_5d,
        ROUND((f20.close - d.close) / d.close * 100, 4) AS fwd_ret_20d,

        -- direction label (for classification)
        CASE WHEN f1.close > d.close THEN 1 ELSE 0 END AS up_1d

    FROM daily d
    LEFT JOIN daily f1
        ON f1.trade_date = (
            SELECT MIN(trade_date) FROM daily
            WHERE trade_date > d.trade_date
        )
    LEFT JOIN daily f5
        ON f5.trade_date = (
            SELECT trade_date FROM daily
            WHERE trade_date > d.trade_date
            ORDER BY trade_date LIMIT 1 OFFSET 4
        )
    LEFT JOIN daily f20
        ON f20.trade_date = (
            SELECT trade_date FROM daily
            WHERE trade_date > d.trade_date
            ORDER BY trade_date LIMIT 1 OFFSET 19
        )
    ;
""")

In [5]:
results = con.execute(""" SELECT * FROM forward_returns ORDER BY trade_date ASC LIMIT 10 OFFSET 20; """).fetchall()
for row in results:
    print(row)

(datetime.date(2016, 2, 1), 'Nifty 50', 7555.95, -1.3288, -2.2327, -7.5292, 0)
(datetime.date(2016, 2, 2), 'Nifty 50', 7455.55, -1.2575, -2.1105, -3.1285, 0)
(datetime.date(2016, 2, 3), 'Nifty 50', 7361.8, 0.5732, -1.9846, 0.0958, 1)
(datetime.date(2016, 2, 4), 'Nifty 50', 7404.0, 1.1494, -5.7759, 0.967, 1)
(datetime.date(2016, 2, 5), 'Nifty 50', 7489.1, -1.36, -6.7852, -0.0501, 0)
(datetime.date(2016, 2, 8), 'Nifty 50', 7387.25, -1.2055, -3.0363, 1.3273, 0)
(datetime.date(2016, 2, 9), 'Nifty 50', 7298.2, -1.1304, -3.4248, 3.2008, 0)
(datetime.date(2016, 2, 10), 'Nifty 50', 7215.7, -3.3171, -1.4863, 3.7481, 0)
(datetime.date(2016, 2, 11), 'Nifty 50', 6976.35, 0.0659, 3.0876, 7.6523, 1)
(datetime.date(2016, 2, 12), 'Nifty 50', 6980.95, 2.6071, 3.2918, 7.9903, 1)


In [6]:
results = con.execute(""" SELECT DISTINCT index_name FROM nse.market_activity_index ORDER BY 1; """).fetchall()
for row in results:
    print(row)

('BHARATBOND-APR25',)
('BHARATBOND-APR30',)
('BHARATBOND-APR31',)
('BHARATBOND-APR32',)
('BHARATBOND-APR33',)
('India VIX',)
('NIFTY Alpha 50',)
('NIFTY AlphaLowVol',)
('NIFTY CONSR DURBL',)
('NIFTY HEALTHCARE',)
('NIFTY IND DIGITAL',)
('NIFTY INDIA MFG',)
('NIFTY LARGEMID250',)
('NIFTY M150 QLTY50',)
('NIFTY MICROCAP250',)
('NIFTY MID SELECT',)
('NIFTY MIDCAP 100',)
('NIFTY MIDCAP 150',)
('NIFTY MIDSML 400',)
('NIFTY OIL AND GAS',)
('NIFTY SMLCAP 100',)
('NIFTY SMLCAP 250',)
('NIFTY SMLCAP 50',)
('NIFTY TOTAL MKT',)
('NIFTY100 EQL Wgt',)
('NIFTY100 ESG',)
('NIFTY100 LowVol30',)
('NIFTY100 Qualty30',)
('NIFTY200 QUALTY30',)
('NIFTY50 EQL Wgt',)
('NIFTY500 MULTICAP',)
('Nifty 100',)
('Nifty 200',)
('Nifty 50',)
('Nifty 500',)
('Nifty AQL 30',)
('Nifty AQLV 30',)
('Nifty Auto',)
('Nifty Bank',)
('Nifty CPSE',)
('Nifty Capital Mkt',)
('Nifty Cement',)
('Nifty Chemicals',)
('Nifty Commodities',)
('Nifty Consumption',)
('Nifty CoreHousing',)
('Nifty Corp MAATR',)
('Nifty Div Opps 50',)
('Ni

In [7]:
con.execute("""
CREATE OR REPLACE TABLE daily_features AS
SELECT
    fr.trade_date,
    fr.close         AS nifty_close,
    fr.fwd_ret_1d,
    fr.fwd_ret_5d,
    fr.fwd_ret_20d,
    fr.up_1d,
    vix.close        AS vix_close

FROM forward_returns fr
LEFT JOIN nse.market_activity_index vix
    ON vix.trade_date = fr.trade_date
    AND vix.index_name = 'India VIX'

ORDER BY fr.trade_date
""")

In [8]:
results = con.execute("""
    SELECT * FROM daily_features
    WHERE vix_close IS NOT NULL
    ORDER BY trade_date DESC LIMIT 10 OFFSET 20
""").fetchall()
for row in results: print(row)

(datetime.date(2026, 6, 24), 24021.65, 0.143, 0.6413, -0.633, 1, 13.385)
(datetime.date(2026, 6, 23), 23824.1, 0.8292, 0.7629, 0.7226, 1, 13.9425)
(datetime.date(2026, 6, 22), 24102.9, -1.1567, -0.9839, 0.3518, 0, 12.8425)
(datetime.date(2026, 6, 19), 24013.1, 0.374, -0.2784, 0.9387, 1, 12.97)
(datetime.date(2026, 6, 18), 24168.0, -0.6409, -0.4634, 0.6881, 0, 12.6725)
(datetime.date(2026, 6, 17), 24085.7, 0.3417, -0.2659, -0.0538, 1, 13.1875)
(datetime.date(2026, 6, 16), 23989.15, 0.4025, -0.688, 0.3725, 1, 13.3625)
(datetime.date(2026, 6, 15), 23853.9, 0.567, 1.0439, 0.8307, 1, 14.3525)
(datetime.date(2026, 6, 12), 23622.9, 0.9779, 1.6518, 2.4895, 1, 14.7175)
(datetime.date(2026, 6, 11), 23161.6, 1.9917, 4.3451, 4.5131, 1, 15.6125)


In [9]:
results = con.execute("""
    SELECT
        CASE
            WHEN vix_close < 13 THEN '1_low <13'
            WHEN vix_close < 16 THEN '2_calm 13-16'
            WHEN vix_close < 20 THEN '3_normal 16-20'
            WHEN vix_close < 25 THEN '4_elevated 20-25'
            ELSE                     '5_fear >25'
        END AS vix_regime,

        COUNT(*)                        AS days,
        ROUND(AVG(fwd_ret_1d), 3)       AS avg_ret_1d,
        ROUND(AVG(fwd_ret_5d), 3)       AS avg_ret_5d,
        ROUND(AVG(fwd_ret_20d), 3)      AS avg_ret_20d,
        ROUND(AVG(up_1d) * 100, 1)      AS pct_up_days

    FROM daily_features
    WHERE fwd_ret_1d IS NOT NULL

    GROUP BY 1
    ORDER BY 1
""").fetchall()

for row in results: print(row)

('1_low <13', 650, 0.053, 0.165, 1.037, 54.9)
('2_calm 13-16', 886, 0.015, 0.167, 0.178, 52.8)
('3_normal 16-20', 626, 0.012, 0.148, 0.609, 54.3)
('4_elevated 20-25', 320, 0.114, 0.441, 2.291, 53.1)
('5_fear >25', 132, 0.242, 1.102, 5.171, 60.6)


In [10]:
results = con.execute("""
    SELECT *
    FROM nse.options_analytics
    WHERE ticker = 'NIFTY'
    ORDER BY trade_date DESC, expiry ASC
    LIMIT 20
""").fetchall()
for row in results: print(row)

('IDO', 'NIFTY', datetime.date(2026, 7, 28), datetime.date(2026, 7, 23), 139967880.0, 206256635.0, 0.6786103147663589, 24000.0)
('IDO', 'NIFTY', datetime.date(2026, 8, 4), datetime.date(2026, 7, 23), 15831725.0, 17647110.0, 0.8971284816607366, 24000.0)
('IDO', 'NIFTY', datetime.date(2026, 8, 11), datetime.date(2026, 7, 23), 1862575.0, 2717520.0, 0.6853951396861845, 24000.0)
('IDO', 'NIFTY', datetime.date(2026, 8, 18), datetime.date(2026, 7, 23), 298805.0, 372320.0, 0.8025488826815642, 24000.0)
('IDO', 'NIFTY', datetime.date(2026, 8, 25), datetime.date(2026, 7, 23), 31583695.0, 29536390.0, 1.0693146657394488, 24100.0)
('IDO', 'NIFTY', datetime.date(2026, 9, 1), datetime.date(2026, 7, 23), 11700.0, 8190.0, 1.4285714285714286, 24100.0)
('IDO', 'NIFTY', datetime.date(2026, 9, 29), datetime.date(2026, 7, 23), 17084475.0, 10709175.0, 1.595311963806736, 24800.0)
('IDO', 'NIFTY', datetime.date(2026, 12, 29), datetime.date(2026, 7, 23), 15226110.0, 12291755.0, 1.2387254708542434, 25000.0)
('IDO

In [11]:
results = con.execute("""
    SELECT
        expiry,
        trade_date,
        pe_oi,
        ce_oi,
        pe_oi + ce_oi AS total_oi
    FROM nse.options_analytics
    WHERE ticker = 'NIFTY'
      AND trade_date = '2026-06-10'
    ORDER BY total_oi DESC
""").fetchall()
for row in results: print(row)

(datetime.date(2026, 6, 16), datetime.date(2026, 6, 10), 93749955.0, 114460970.0, 208210925.0)
(datetime.date(2026, 6, 30), datetime.date(2026, 6, 10), 69249000.0, 66301370.0, 135550370.0)
(datetime.date(2026, 12, 29), datetime.date(2026, 6, 10), 11812480.0, 9371595.0, 21184075.0)
(datetime.date(2026, 6, 23), datetime.date(2026, 6, 10), 9232405.0, 10127780.0, 19360185.0)
(datetime.date(2026, 7, 28), datetime.date(2026, 6, 10), 9825335.0, 8697390.0, 18522725.0)
(datetime.date(2026, 9, 29), datetime.date(2026, 6, 10), 5249010.0, 4500755.0, 9749765.0)
(datetime.date(2026, 8, 25), datetime.date(2026, 6, 10), 2054260.0, 1737060.0, 3791320.0)
(datetime.date(2026, 7, 7), datetime.date(2026, 6, 10), 275860.0, 505245.0, 781105.0)
(datetime.date(2027, 12, 28), datetime.date(2026, 6, 10), 218430.0, 155535.0, 373965.0)
(datetime.date(2028, 12, 26), datetime.date(2026, 6, 10), 44980.0, 16115.0, 61095.0)
(datetime.date(2026, 7, 14), datetime.date(2026, 6, 10), 9425.0, 12155.0, 21580.0)
(datetime.dat

In [12]:
results = con.execute("""
    SELECT
        percentile_cont(0.25) WITHIN GROUP (ORDER BY total_oi) AS p25,
        percentile_cont(0.50) WITHIN GROUP (ORDER BY total_oi) AS p50,
        percentile_cont(0.75) WITHIN GROUP (ORDER BY total_oi) AS p75,
        percentile_cont(0.90) WITHIN GROUP (ORDER BY total_oi) AS p90,
        percentile_cont(0.95) WITHIN GROUP (ORDER BY total_oi) AS p95,
        MIN(total_oi)  AS min_oi,
        MAX(total_oi)  AS max_oi,
        COUNT(*)       AS total_rows
    FROM (
        SELECT pe_oi + ce_oi AS total_oi
        FROM nse.options_analytics
        WHERE ticker = 'NIFTY'
          AND pe_oi IS NOT NULL
          AND ce_oi IS NOT NULL
    )
""").fetchall()
for row in results: print(row)

(1600.0, 434925.0, 5765350.0, 34815340.00000009, 78585479.99999997, 0.0, 424741950.0, 46869)


In [13]:
results = con.execute("""
    SELECT
        trade_date,
        COUNT(*) AS liquid_expiries
    FROM nse.options_analytics
    WHERE ticker = 'NIFTY'
      AND (pe_oi + ce_oi) > 11800000
    GROUP BY trade_date
    ORDER BY trade_date DESC
    LIMIT 20
""").fetchall()
for row in results: print(row)

(datetime.date(2026, 7, 23), 5)
(datetime.date(2026, 7, 22), 5)
(datetime.date(2026, 7, 21), 6)
(datetime.date(2026, 7, 20), 5)
(datetime.date(2026, 7, 17), 5)
(datetime.date(2026, 7, 16), 5)
(datetime.date(2026, 7, 15), 5)
(datetime.date(2026, 7, 14), 6)
(datetime.date(2026, 7, 13), 6)
(datetime.date(2026, 7, 10), 6)
(datetime.date(2026, 7, 9), 6)
(datetime.date(2026, 7, 8), 6)
(datetime.date(2026, 7, 7), 6)
(datetime.date(2026, 7, 6), 6)
(datetime.date(2026, 7, 3), 6)
(datetime.date(2026, 7, 2), 6)
(datetime.date(2026, 7, 1), 6)
(datetime.date(2026, 6, 30), 6)
(datetime.date(2026, 6, 29), 5)
(datetime.date(2026, 6, 25), 5)


In [14]:
con.execute("""
CREATE OR REPLACE TABLE daily_features AS
SELECT
    fr.trade_date,
    fr.close              AS nifty_close,
    fr.fwd_ret_1d,
    fr.fwd_ret_5d,
    fr.fwd_ret_20d,
    fr.up_1d,
    vix.close             AS vix_close,
    pcr_agg.pcr,
    mp_agg.max_pain_dist_pct

FROM forward_returns fr

-- VIX
LEFT JOIN nse.market_activity_index vix
    ON vix.trade_date = fr.trade_date
    AND vix.index_name = 'India VIX'

-- FO volatility
LEFT JOIN nse.fo_volatility fv
    ON fv.trade_date = fr.trade_date
    AND fv.ticker = 'NIFTY'

-- Market breadth
LEFT JOIN nse.market_activity_breadth mab
    ON mab.trade_date = fr.trade_date

-- Market activity summary
LEFT JOIN nse.market_activity_summary mas
    ON mas.trade_date = fr.trade_date

-- PCR: all expiries
LEFT JOIN (
    SELECT
        trade_date,
        ROUND(SUM(pe_oi) / NULLIF(SUM(ce_oi), 0), 4) AS pcr
    FROM nse.options_analytics
    WHERE ticker = 'NIFTY'
      AND instrument_type = 'IDO'
    GROUP BY trade_date
) pcr_agg ON pcr_agg.trade_date = fr.trade_date

-- Max pain distance: liquid expiries only, OI-weighted
LEFT JOIN (
    SELECT oa.trade_date,
        ROUND(
            SUM(((fr2.close - oa.max_pain) / NULLIF(oa.max_pain, 0) * 100) * (oa.pe_oi + oa.ce_oi))
            / NULLIF(SUM(oa.pe_oi + oa.ce_oi), 0)
        , 4) AS max_pain_dist_pct
    FROM nse.options_analytics oa
    JOIN forward_returns fr2 ON fr2.trade_date = oa.trade_date
    WHERE oa.ticker = 'NIFTY' AND oa.instrument_type = 'IDO'
      AND (oa.pe_oi + oa.ce_oi) > 11800000
    GROUP BY oa.trade_date
) mp_agg ON mp_agg.trade_date = fr.trade_date

ORDER BY fr.trade_date
""")

In [16]:
results = con.execute("""
    SELECT * FROM daily_features
    WHERE vix_close IS NOT NULL
    ORDER BY trade_date DESC LIMIT 10
""").fetchall()
for row in results: print(row)

(datetime.date(2026, 7, 23), 23869.6, None, None, None, 0, 13.475, 0.7966, -0.9946)
(datetime.date(2026, 7, 22), 23996.25, -0.5278, None, None, 0, 13.2925, 0.8438, -0.9417)
(datetime.date(2026, 7, 21), 24187.7, -0.7915, None, None, 0, 12.6, 0.9065, -0.3217)
(datetime.date(2026, 7, 20), 24238.5, -0.2096, None, None, 0, 12.98, 1.2764, -0.2196)
(datetime.date(2026, 7, 17), 24334.3, -0.3937, None, None, 0, 13.15, 1.4234, 0.0427)
(datetime.date(2026, 7, 16), 24072.75, 1.0865, -0.8439, None, 1, 12.8825, 0.97, -0.748)
(datetime.date(2026, 7, 15), 24078.5, -0.0239, -0.3416, None, 0, 13.27, 0.9475, -0.7651)
(datetime.date(2026, 7, 14), 24052.05, 0.11, 0.564, None, 1, 13.75, 1.012, -0.4381)
(datetime.date(2026, 7, 13), 24211.0, -0.6565, 0.1136, None, 0, 13.28, 1.4303, -0.2571)
(datetime.date(2026, 7, 10), 24206.9, 0.0169, 0.5263, None, 1, 12.2525, 1.2499, -0.3108)


In [17]:
results = con.execute("""
    SELECT
        CASE
            WHEN pcr < 0.7  THEN '1_very_low <0.7'
            WHEN pcr < 0.9  THEN '2_low 0.7-0.9'
            WHEN pcr < 1.1  THEN '3_neutral 0.9-1.1'
            WHEN pcr < 1.3  THEN '4_high 1.1-1.3'
            ELSE                 '5_very_high >1.3'
        END AS pcr_bucket,

        COUNT(*)                    AS days,
        ROUND(AVG(fwd_ret_1d), 3)   AS avg_ret_1d,
        ROUND(AVG(fwd_ret_5d), 3)   AS avg_ret_5d,
        ROUND(AVG(fwd_ret_20d), 3)  AS avg_ret_20d,
        ROUND(AVG(up_1d) * 100, 1)  AS pct_up_days

    FROM daily_features
    WHERE fwd_ret_1d IS NOT NULL AND pcr IS NOT NULL
    GROUP BY 1
    ORDER BY 1
""").fetchall()

print("=== PCR ===")
for row in results: print(row)

results = con.execute("""
    SELECT
        CASE
            WHEN max_pain_dist_pct < -3   THEN '1_far_below <-3%'
            WHEN max_pain_dist_pct < -1.5 THEN '2_below -3 to -1.5%'
            WHEN max_pain_dist_pct < 0    THEN '3_slightly_below -1.5 to 0%'
            WHEN max_pain_dist_pct < 1.5  THEN '4_slightly_above 0 to 1.5%'
            ELSE                               '5_far_above >1.5%'
        END AS mp_bucket,

        COUNT(*)                    AS days,
        ROUND(AVG(fwd_ret_1d), 3)   AS avg_ret_1d,
        ROUND(AVG(fwd_ret_5d), 3)   AS avg_ret_5d,
        ROUND(AVG(fwd_ret_20d), 3)  AS avg_ret_20d,
        ROUND(AVG(up_1d) * 100, 1)  AS pct_up_days

    FROM daily_features
    WHERE fwd_ret_1d IS NOT NULL AND max_pain_dist_pct IS NOT NULL
    GROUP BY 1
    ORDER BY 1
""").fetchall()

print("\n=== Max Pain Distance ===")
for row in results: print(row)

=== PCR ===
('1_very_low <0.7', 55, 0.025, 0.577, 0.753, 52.7)
('2_low 0.7-0.9', 459, 0.035, 0.154, 0.622, 48.6)
('3_neutral 0.9-1.1', 719, 0.031, 0.243, 1.002, 53.1)
('4_high 1.1-1.3', 657, -0.005, 0.175, 1.116, 52.7)
('5_very_high >1.3', 724, 0.121, 0.335, 1.184, 60.1)

=== Max Pain Distance ===
('1_far_below <-3%', 5, 0.292, 0.385, 5.101, 80.0)
('2_below -3 to -1.5%', 29, 0.107, 0.588, 2.062, 69.0)
('3_slightly_below -1.5 to 0%', 309, -0.002, 0.073, -0.132, 46.9)
('4_slightly_above 0 to 1.5%', 172, 0.004, -0.153, -0.169, 54.1)
('5_far_above >1.5%', 1, -1.39, 0.083, 0.716, 0.0)


In [18]:
# Futures - what tickers and how many rows
results = con.execute("""
    SELECT instrument_type, ticker, COUNT(*) as rows, 
           MIN(trade_date) as from_date, MAX(trade_date) as to_date
    FROM nse.futures_analytics
    GROUP BY instrument_type, ticker
    ORDER BY rows DESC
    LIMIT 10
""").fetchall()
print("=== Futures ===")
for row in results: print(row)

=== Futures ===
('STF', 'ONGC', 7845, datetime.date(2016, 1, 1), datetime.date(2026, 7, 23))
('STF', 'COALINDIA', 7845, datetime.date(2016, 1, 1), datetime.date(2026, 7, 23))
('STF', 'M&M', 7845, datetime.date(2016, 1, 1), datetime.date(2026, 7, 23))
('STF', 'KOTAKBANK', 7845, datetime.date(2016, 1, 1), datetime.date(2026, 7, 23))
('STF', 'COLPAL', 7845, datetime.date(2016, 1, 1), datetime.date(2026, 7, 23))
('STF', 'INFY', 7845, datetime.date(2016, 1, 1), datetime.date(2026, 7, 23))
('STF', 'HEROMOTOCO', 7845, datetime.date(2016, 1, 1), datetime.date(2026, 7, 23))
('STF', 'IDEA', 7845, datetime.date(2016, 1, 1), datetime.date(2026, 7, 23))
('STF', 'TATASTEEL', 7845, datetime.date(2016, 1, 1), datetime.date(2026, 7, 23))
('STF', 'DLF', 7845, datetime.date(2016, 1, 1), datetime.date(2026, 7, 23))


In [19]:
# Participant - what participant types and asset classes exist
results = con.execute("""
    SELECT participant_type, metric_type, asset_class, direction, option_side,
           COUNT(*) as rows
    FROM nse.participant_activity
    GROUP BY participant_type, metric_type, asset_class, direction, option_side
    ORDER BY participant_type, metric_type, asset_class
    LIMIT 30
""").fetchall()
print("\n=== Participant ===")
for row in results: print(row)


=== Participant ===
('Client', 'OI', 'INDEX', 'long', 'NA', 2614)
('Client', 'OI', 'INDEX', 'long', 'CE', 2614)
('Client', 'OI', 'INDEX', 'short', 'PE', 2614)
('Client', 'OI', 'INDEX', 'long', 'PE', 2614)
('Client', 'OI', 'INDEX', 'short', 'CE', 2614)
('Client', 'OI', 'INDEX', 'short', 'NA', 2614)
('Client', 'OI', 'STOCK', 'short', 'NA', 2614)
('Client', 'OI', 'STOCK', 'short', 'CE', 2614)
('Client', 'OI', 'STOCK', 'short', 'PE', 2614)
('Client', 'OI', 'STOCK', 'long', 'CE', 2614)
('Client', 'OI', 'STOCK', 'long', 'NA', 2614)
('Client', 'OI', 'STOCK', 'long', 'PE', 2614)
('Client', 'VOL', 'INDEX', 'long', 'NA', 2614)
('Client', 'VOL', 'INDEX', 'short', 'NA', 2614)
('Client', 'VOL', 'INDEX', 'long', 'PE', 2614)
('Client', 'VOL', 'INDEX', 'short', 'PE', 2614)
('Client', 'VOL', 'INDEX', 'long', 'CE', 2614)
('Client', 'VOL', 'INDEX', 'short', 'CE', 2614)
('Client', 'VOL', 'STOCK', 'short', 'CE', 2614)
('Client', 'VOL', 'STOCK', 'long', 'NA', 2614)
('Client', 'VOL', 'STOCK', 'short', 'PE',

In [20]:
results = con.execute("""
  SELECT fa.trade_date, fa.expiry, fa.basis, fa.cost_of_carry, 
        fa.chng_oi_per, mdd.open_interest
  FROM nse.futures_analytics fa
  JOIN nse.instruments i
      ON i.ticker = fa.ticker
      AND i.instrument_type = fa.instrument_type
      AND i.expiry = fa.expiry
  JOIN nse.market_data_daily mdd
      ON mdd.instrument_key = i.instrument_key
      AND mdd.trade_date = fa.trade_date
  WHERE fa.ticker = 'NIFTY'
    AND fa.instrument_type = 'IDF'
    AND fa.trade_date = '2026-06-10'
  ORDER BY fa.expiry ASC
""").fetchall()
for row in results: print(row)

(datetime.date(2026, 6, 10), datetime.date(2026, 6, 30), 25.149999999997817, 0.019771203470175906, -1.4561936771066268, 19134310)
(datetime.date(2026, 6, 10), datetime.date(2026, 7, 28), 129.95000000000073, 0.042565737093267005, 0.20905923344947736, 1682460)
(datetime.date(2026, 6, 10), datetime.date(2026, 8, 25), 232.45000000000073, 0.04808848222918073, 1.3488880787458988, 542100)


In [ ]:
con.execute("""
CREATE OR REPLACE TABLE daily_features AS
SELECT
    fr.trade_date,
    fr.close             AS nifty_close,
    fr.fwd_ret_1d,
    fr.fwd_ret_5d,
    fr.fwd_ret_20d,
    fr.up_1d,

    -- VIX
    vix.close            AS vix_close,

    -- PCR (all expiries)
    pcr_agg.pcr,

    -- Max pain distance (liquid expiries, OI-weighted)
    mp_agg.max_pain_dist_pct,

    -- Futures near month
    fut.basis,
    fut.cost_of_carry,
    fut.chng_oi_per      AS fut_chng_oi_pct,
    
    fv.underlying_daily_vol,
    fv.futures_daily_vol,
    fv.applicable_daily_vol,

    -- Breadth
    mab.advances,
    mab.declines,
    ROUND(mab.advances::DOUBLE / NULLIF(mab.declines, 0), 4) AS adv_decl_ratio,
    mab.price_band_hits,

    -- Market summary
    mas.traded_value_cr,
    mas.num_trades,
    mas.market_cap_cr,

    -- FII net futures positioning (long - short, futures only)
    ROUND(
        (fii_long_fut.contracts - fii_short_fut.contracts) /
        NULLIF(fii_long_fut.contracts + fii_short_fut.contracts, 0) * 100
    , 2)                 AS fii_fut_net_pct,

    -- Client net futures positioning (retail, often contrarian)
    ROUND(
        (cli_long_fut.contracts - cli_short_fut.contracts) /
        NULLIF(cli_long_fut.contracts + cli_short_fut.contracts, 0) * 100
    , 2)                 AS client_fut_net_pct,

    -- FII put buying (protection signal)
    ROUND(
        (fii_long_pe.contracts - fii_short_pe.contracts) /
        NULLIF(fii_long_pe.contracts + fii_short_pe.contracts, 0) * 100
    , 2)                 AS fii_pe_net_pct,

    fii_flow.fii_net_flow_cr

FROM forward_returns fr

-- VIX
LEFT JOIN nse.market_activity_index vix
    ON vix.trade_date = fr.trade_date
    AND vix.index_name = 'India VIX'

-- FO volatility
LEFT JOIN nse.fo_volatility fv
    ON fv.trade_date = fr.trade_date
    AND fv.ticker = 'NIFTY'

-- Market breadth
LEFT JOIN nse.market_activity_breadth mab
    ON mab.trade_date = fr.trade_date

-- Market activity summary
LEFT JOIN nse.market_activity_summary mas
    ON mas.trade_date = fr.trade_date

-- PCR: all expiries
LEFT JOIN (
    SELECT trade_date,
        ROUND(SUM(pe_oi) / NULLIF(SUM(ce_oi), 0), 4) AS pcr
    FROM nse.options_analytics
    WHERE ticker = 'NIFTY' AND instrument_type = 'IDO'
    GROUP BY trade_date
) pcr_agg ON pcr_agg.trade_date = fr.trade_date

-- Max pain distance: liquid expiries only, OI-weighted
LEFT JOIN (
    SELECT oa.trade_date,
        ROUND(
            SUM(((fr2.close - oa.max_pain) / NULLIF(oa.max_pain, 0) * 100) * (oa.pe_oi + oa.ce_oi))
            / NULLIF(SUM(oa.pe_oi + oa.ce_oi), 0)
        , 4) AS max_pain_dist_pct
    FROM nse.options_analytics oa
    JOIN forward_returns fr2 ON fr2.trade_date = oa.trade_date
    WHERE oa.ticker = 'NIFTY' AND oa.instrument_type = 'IDO'
      AND (oa.pe_oi + oa.ce_oi) > 11800000
    GROUP BY oa.trade_date
) mp_agg ON mp_agg.trade_date = fr.trade_date

-- Futures: near month only
LEFT JOIN (
    SELECT fa.trade_date, fa.basis, fa.cost_of_carry, fa.chng_oi_per,
           mdd.open_interest AS fut_open_interest
    FROM nse.futures_analytics fa
    JOIN nse.instruments i
        ON i.ticker = fa.ticker AND i.instrument_type = fa.instrument_type AND i.expiry = fa.expiry
    JOIN nse.market_data_daily mdd
        ON mdd.instrument_key = i.instrument_key AND mdd.trade_date = fa.trade_date
    WHERE fa.ticker = 'NIFTY' AND fa.instrument_type = 'IDF'
      AND (fa.trade_date, fa.expiry) IN (
          SELECT trade_date, MIN(expiry)
          FROM nse.futures_analytics
          WHERE ticker = 'NIFTY' AND instrument_type = 'IDF'
          GROUP BY trade_date
      )
) fut ON fut.trade_date = fr.trade_date

-- Participant: FII long futures
LEFT JOIN (
    SELECT trade_date, contracts FROM nse.participant_activity
    WHERE participant_type = 'FII' AND metric_type = 'OI'
      AND asset_class = 'INDEX' AND direction = 'long' AND option_side = 'NA'
) fii_long_fut ON fii_long_fut.trade_date = fr.trade_date

-- Participant: FII short futures
LEFT JOIN (
    SELECT trade_date, contracts FROM nse.participant_activity
    WHERE participant_type = 'FII' AND metric_type = 'OI'
      AND asset_class = 'INDEX' AND direction = 'short' AND option_side = 'NA'
) fii_short_fut ON fii_short_fut.trade_date = fr.trade_date

-- Participant: Client long futures
LEFT JOIN (
    SELECT trade_date, contracts FROM nse.participant_activity
    WHERE participant_type = 'Client' AND metric_type = 'OI'
      AND asset_class = 'INDEX' AND direction = 'long' AND option_side = 'NA'
) cli_long_fut ON cli_long_fut.trade_date = fr.trade_date

-- Participant: Client short futures
LEFT JOIN (
    SELECT trade_date, contracts FROM nse.participant_activity
    WHERE participant_type = 'Client' AND metric_type = 'OI'
      AND asset_class = 'INDEX' AND direction = 'short' AND option_side = 'NA'
) cli_short_fut ON cli_short_fut.trade_date = fr.trade_date

-- Participant: FII long PE (put buying = hedging/bearish)
LEFT JOIN (
    SELECT trade_date, contracts FROM nse.participant_activity
    WHERE participant_type = 'FII' AND metric_type = 'OI'
      AND asset_class = 'INDEX' AND direction = 'long' AND option_side = 'PE'
) fii_long_pe ON fii_long_pe.trade_date = fr.trade_date

-- Participant: FII short PE
LEFT JOIN (
    SELECT trade_date, contracts FROM nse.participant_activity
    WHERE participant_type = 'FII' AND metric_type = 'OI'
      AND asset_class = 'INDEX' AND direction = 'short' AND option_side = 'PE'
) fii_short_pe ON fii_short_pe.trade_date = fr.trade_date

-- FII stats: net flow in Cr (index futures)
LEFT JOIN (
    SELECT trade_date, buy_amount_cr, sell_amount_cr,
           ROUND(buy_amount_cr - sell_amount_cr, 2) AS fii_net_flow_cr
    FROM nse.fii_stats
    WHERE instrument = 'INDEX FUTURES'
) fii_flow ON fii_flow.trade_date = fr.trade_date

ORDER BY fr.trade_date
""")

In [22]:
con.execute("""
    UPDATE daily_features
    SET underlying_daily_vol = (
            SELECT AVG(underlying_daily_vol) FROM daily_features
            WHERE trade_date IN ('2021-03-30', '2021-04-01')
        ),
        futures_daily_vol = (
            SELECT AVG(futures_daily_vol) FROM daily_features
            WHERE trade_date IN ('2021-03-30', '2021-04-01')
        ),
        applicable_daily_vol = (
            SELECT AVG(applicable_daily_vol) FROM daily_features
            WHERE trade_date IN ('2021-03-30', '2021-04-01')
        )
    WHERE trade_date = '2021-03-31'
      AND underlying_daily_vol IS NULL
""")

In [23]:
con.execute("""
    UPDATE daily_features
    SET fii_fut_net_pct = (
            SELECT AVG(fii_fut_net_pct) FROM daily_features
            WHERE trade_date IN ('2016-02-09', '2016-02-11')
        ),
        client_fut_net_pct = (
            SELECT AVG(client_fut_net_pct) FROM daily_features
            WHERE trade_date IN ('2016-02-09', '2016-02-11')
        ),
        fii_pe_net_pct = (
            SELECT AVG(fii_pe_net_pct) FROM daily_features
            WHERE trade_date IN ('2016-02-09', '2016-02-11')
        )
    WHERE trade_date = '2016-02-10'
      AND fii_fut_net_pct IS NULL
""")

In [25]:
results = con.execute("""
    SELECT trade_date, underlying_daily_vol, futures_daily_vol, applicable_daily_vol
    FROM daily_features
    WHERE trade_date BETWEEN '2016-02-08' AND '2016-02-12'
    ORDER BY trade_date
""").fetchall()
for row in results: print(row)

(datetime.date(2016, 2, 8), 0.0107, 0.0107, 0.0107)
(datetime.date(2016, 2, 9), 0.0108, 0.0107, 0.0108)
(datetime.date(2016, 2, 10), 0.0108, 0.0107, 0.0108)
(datetime.date(2016, 2, 11), 0.0133, 0.0135, 0.0135)
(datetime.date(2016, 2, 12), 0.0129, 0.0131, 0.0131)


In [26]:
results = con.execute("""
    SELECT trade_date, fii_fut_net_pct, client_fut_net_pct, fii_pe_net_pct
    FROM daily_features
    WHERE trade_date BETWEEN '2016-02-08' AND '2016-02-12'
    ORDER BY trade_date
""").fetchall()
for row in results: print(row)

(datetime.date(2016, 2, 8), -17.07, -0.07, 66.11)
(datetime.date(2016, 2, 9), -12.57, -1.54, 70.85)
(datetime.date(2016, 2, 10), -12.105, 2.33, 72.715)
(datetime.date(2016, 2, 11), -11.64, 6.2, 74.58)
(datetime.date(2016, 2, 12), -4.81, 4.1, 73.11)


In [27]:
results = con.execute("""
    SELECT * FROM daily_features
    WHERE vix_close IS NOT NULL
    ORDER BY trade_date DESC LIMIT 5
""").fetchall()
for row in results: print(row)

(datetime.date(2026, 7, 23), 23869.6, None, None, None, 0, 13.475, 0.7966, -0.9946, 4.0, 0.012233133357911319, -18.273668678980467, 0.00881082, 0.00888588, 0.00888588, 1154, 2292, 0.5035, 188, 111749.62, 35892377, 47572808.85, -84.42, 56.14, 35.75, -1840.02)
(datetime.date(2026, 7, 22), 23996.25, -0.5278, None, None, 0, 13.2925, 0.8438, -0.9417, -7.849999999998545, -0.019900678925522868, 0.8108475806085891, 0.00882474, 0.00890179, 0.00890179, 1147, 2350, 0.4881, 178, 111459.97, 35320601, 47909998.2, -83.74, 56.5, 36.93, -3606.85)
(datetime.date(2026, 7, 21), 24187.7, -0.7915, None, None, 0, 12.6, 0.9065, -0.3217, -7.100000000002183, -0.015305890420106069, 1.613765391137654, 0.00882895, 0.00890644, 0.00890644, 1883, 1630, 1.1552, 186, 113350.33, 35355780, 48343129.98, -82.83, 52.55, 41.23, -1487.46)
(datetime.date(2026, 7, 20), 24238.5, -0.2096, None, None, 0, 12.98, 1.2764, -0.2196, 21.0, 0.03952905501578068, -3.5909920876445525, 0.00885012, 0.00892595, 0.00892595, 1992, 1642, 1.2132, 

In [28]:
queries = {
    "Basis": ("basis", [
        ("1_discount <0",      "basis < 0"),
        ("2_low 0-50",         "basis >= 0 AND basis < 50"),
        ("3_mid 50-100",       "basis >= 50 AND basis < 100"),
        ("4_high >100",        "basis >= 100"),
    ]),
    "FII Fut Net": ("fii_fut_net_pct", [
        ("1_very_short <-50",  "fii_fut_net_pct < -50"),
        ("2_short -50 to -20", "fii_fut_net_pct >= -50 AND fii_fut_net_pct < -20"),
        ("3_neutral -20 to 20","fii_fut_net_pct >= -20 AND fii_fut_net_pct < 20"),
        ("4_long 20 to 50",    "fii_fut_net_pct >= 20 AND fii_fut_net_pct < 50"),
        ("5_very_long >50",    "fii_fut_net_pct >= 50"),
    ]),
    "Client Fut Net": ("client_fut_net_pct", [
        ("1_very_short <-50",  "client_fut_net_pct < -50"),
        ("2_short -50 to -20", "client_fut_net_pct >= -50 AND client_fut_net_pct < -20"),
        ("3_neutral -20 to 20","client_fut_net_pct >= -20 AND client_fut_net_pct < 20"),
        ("4_long 20 to 50",    "client_fut_net_pct >= 20 AND client_fut_net_pct < 50"),
        ("5_very_long >50",    "client_fut_net_pct >= 50"),
    ]),
    "FII PE Net": ("fii_pe_net_pct", [
        ("1_very_short <-50",  "fii_pe_net_pct < -50"),
        ("2_short -50 to -20", "fii_pe_net_pct >= -50 AND fii_pe_net_pct < -20"),
        ("3_neutral -20 to 20","fii_pe_net_pct >= -20 AND fii_pe_net_pct < 20"),
        ("4_long 20 to 50",    "fii_pe_net_pct >= 20 AND fii_pe_net_pct < 50"),
        ("5_very_long >50",    "fii_pe_net_pct >= 50"),
    ]),
}

for factor_name, (col, buckets) in queries.items():
    case_sql = "CASE\n" + "\n".join(
        f"  WHEN {cond} THEN '{label}'" for label, cond in buckets
    ) + "\n  ELSE 'other' END"

    results = con.execute(f"""
        SELECT
            {case_sql} AS bucket,
            COUNT(*)                    AS days,
            ROUND(AVG(fwd_ret_1d), 3)   AS avg_ret_1d,
            ROUND(AVG(fwd_ret_5d), 3)   AS avg_ret_5d,
            ROUND(AVG(fwd_ret_20d), 3)  AS avg_ret_20d,
            ROUND(AVG(up_1d) * 100, 1)  AS pct_up_days
        FROM daily_features
        WHERE fwd_ret_1d IS NOT NULL AND {col} IS NOT NULL
        GROUP BY 1 ORDER BY 1
    """).fetchall()

    print(f"\n=== {factor_name} ===")
    for row in results: print(row)


=== Basis ===
('1_discount <0', 52, 0.061, 0.652, 0.134, 55.8)
('2_low 0-50', 172, 0.09, 0.166, 0.254, 54.1)
('3_mid 50-100', 198, -0.061, -0.109, -0.028, 48.0)
('4_high >100', 94, -0.037, -0.252, -0.273, 47.9)

=== FII Fut Net ===
('1_very_short <-50', 521, -0.008, 0.017, 0.097, 50.3)
('2_short -50 to -20', 348, 0.051, 0.178, 1.293, 53.7)
('3_neutral -20 to 20', 946, 0.061, 0.362, 1.223, 54.4)
('4_long 20 to 50', 564, 0.074, 0.263, 1.307, 57.6)
('5_very_long >50', 234, 0.039, 0.298, 0.955, 53.4)

=== Client Fut Net ===
('1_very_short <-50', 29, 0.041, 0.172, -0.381, 55.2)
('2_short -50 to -20', 243, 0.091, 0.237, 1.137, 54.3)
('3_neutral -20 to 20', 1701, 0.068, 0.377, 1.457, 55.1)
('4_long 20 to 50', 595, -0.031, -0.176, -0.323, 50.8)
('5_very_long >50', 46, 0.067, 0.809, 2.299, 58.7)

=== FII PE Net ===
('3_neutral -20 to 20', 597, 0.129, 0.381, 1.159, 60.0)
('4_long 20 to 50', 1698, 0.012, 0.178, 0.91, 52.1)
('5_very_long >50', 318, 0.08, 0.325, 1.248, 53.8)


In [29]:
results = con.execute("""
    SELECT
        CASE
            WHEN vix_close < 14 THEN 'VIX_low'
            WHEN vix_close < 18 THEN 'VIX_mid'
            ELSE                     'VIX_high'
        END AS vix_regime,

        CASE
            WHEN basis < 0   THEN 'basis_discount'
            WHEN basis < 50  THEN 'basis_low'
            WHEN basis < 100 THEN 'basis_mid'
            ELSE                  'basis_high'
        END AS basis_regime,

        COUNT(*)                    AS days,
        ROUND(AVG(fwd_ret_1d), 3)   AS avg_ret_1d,
        ROUND(AVG(fwd_ret_5d), 3)   AS avg_ret_5d,
        ROUND(AVG(fwd_ret_20d), 3)  AS avg_ret_20d,
        ROUND(AVG(up_1d) * 100, 1)  AS pct_up_days

    FROM daily_features
    WHERE fwd_ret_1d IS NOT NULL
      AND vix_close IS NOT NULL
      AND basis IS NOT NULL

    GROUP BY 1, 2
    ORDER BY 1, 2
""").fetchall()

for row in results: print(row)

('VIX_high', 'basis_discount', 11, -0.486, 1.073, 1.22, 54.5)
('VIX_high', 'basis_high', 6, 0.082, 1.017, 0.964, 66.7)
('VIX_high', 'basis_low', 25, 0.375, 0.278, 1.399, 56.0)
('VIX_high', 'basis_mid', 22, 0.226, 1.195, 2.519, 59.1)
('VIX_low', 'basis_discount', 20, 0.15, 0.668, -1.039, 55.0)
('VIX_low', 'basis_high', 62, -0.085, -0.41, -0.297, 45.2)
('VIX_low', 'basis_low', 90, -0.017, -0.045, -0.378, 53.3)
('VIX_low', 'basis_mid', 110, -0.028, -0.133, -0.337, 53.6)
('VIX_mid', 'basis_discount', 21, 0.262, 0.419, 0.347, 57.1)
('VIX_mid', 'basis_high', 26, 0.048, -0.169, -0.5, 50.0)
('VIX_mid', 'basis_low', 57, 0.135, 0.446, 0.669, 54.4)
('VIX_mid', 'basis_mid', 66, -0.21, -0.504, -0.38, 34.8)


In [30]:
results = con.execute("""
    SELECT
        CASE
            WHEN vix_close < 14 THEN 'VIX_low'
            WHEN vix_close < 18 THEN 'VIX_mid'
            ELSE                     'VIX_high'
        END AS vix_regime,

        CASE
            WHEN basis < 0   THEN 'basis_discount'
            WHEN basis < 50  THEN 'basis_low'
            WHEN basis < 100 THEN 'basis_mid'
            ELSE                  'basis_high'
        END AS basis_regime,

        CASE
            WHEN fii_fut_net_pct < -50 THEN 'FII_short'
            WHEN fii_fut_net_pct < 20  THEN 'FII_neutral'
            ELSE                            'FII_long'
        END AS fii_regime,

        COUNT(*)                    AS days,
        ROUND(AVG(fwd_ret_1d), 3)   AS avg_ret_1d,
        ROUND(AVG(fwd_ret_5d), 3)   AS avg_ret_5d,
        ROUND(AVG(fwd_ret_20d), 3)  AS avg_ret_20d,
        ROUND(AVG(up_1d) * 100, 1)  AS pct_up_days

    FROM daily_features
    WHERE fwd_ret_1d IS NOT NULL
      AND vix_close IS NOT NULL
      AND basis IS NOT NULL
      AND fii_fut_net_pct IS NOT NULL

    GROUP BY 1, 2, 3
    HAVING COUNT(*) >= 8      -- filter cells too small to mean anything
    ORDER BY avg_ret_20d DESC
""").fetchall()

for row in results: print(row)

('VIX_high', 'basis_mid', 'FII_neutral', 8, 0.996, 1.524, 3.099, 87.5)
('VIX_high', 'basis_mid', 'FII_short', 14, -0.214, 1.007, 2.188, 42.9)
('VIX_low', 'basis_low', 'FII_neutral', 15, 0.106, -0.135, 1.662, 66.7)
('VIX_high', 'basis_low', 'FII_short', 20, 0.544, 0.376, 1.314, 65.0)
('VIX_high', 'basis_discount', 'FII_short', 10, -0.425, 0.855, 1.145, 60.0)
('VIX_mid', 'basis_low', 'FII_neutral', 27, 0.185, 0.818, 0.852, 63.0)
('VIX_mid', 'basis_low', 'FII_short', 23, 0.092, 0.08, 0.758, 43.5)
('VIX_low', 'basis_mid', 'FII_long', 14, 0.283, -0.027, 0.745, 78.6)
('VIX_mid', 'basis_high', 'FII_short', 17, 0.078, -0.361, 0.255, 47.1)
('VIX_low', 'basis_mid', 'FII_short', 76, -0.023, 0.127, 0.067, 52.6)
('VIX_mid', 'basis_mid', 'FII_short', 46, -0.229, -0.61, -0.044, 34.8)
('VIX_low', 'basis_high', 'FII_short', 49, 0.008, 0.036, -0.113, 49.0)
('VIX_mid', 'basis_discount', 'FII_short', 9, 0.553, -0.09, -0.423, 77.8)
('VIX_low', 'basis_low', 'FII_short', 65, -0.065, -0.044, -0.734, 49.2)
('V

In [31]:
import scipy.stats as stats
import pandas as pd

df = con.execute("""
    SELECT vix_close, pcr, max_pain_dist_pct, basis, 
           cost_of_carry, fut_chng_oi_pct,
           fii_fut_net_pct, client_fut_net_pct,
           underlying_daily_vol, futures_daily_vol, applicable_daily_vol,
           adv_decl_ratio, traded_value_cr, num_trades, market_cap_cr, fii_net_flow_cr,
           fwd_ret_1d, fwd_ret_5d, fwd_ret_20d
    FROM daily_features
    WHERE fwd_ret_1d IS NOT NULL
""").df()

factors = ['vix_close', 'pcr', 'max_pain_dist_pct', 'basis',
           'cost_of_carry', 'fut_chng_oi_pct', 'fii_fut_net_pct', 
           'client_fut_net_pct', 'underlying_daily_vol', 
           'futures_daily_vol', 'applicable_daily_vol',
           'adv_decl_ratio', 'traded_value_cr', 'num_trades', 'market_cap_cr', 'fii_net_flow_cr']

targets = ['fwd_ret_1d', 'fwd_ret_5d', 'fwd_ret_20d']

rows = []
for f in factors:
    row = {'factor': f}
    for t in targets:
        mask = df[f].notna() & df[t].notna()
        ic, pval = stats.spearmanr(df.loc[mask, f], df.loc[mask, t])
        row[f'IC_{t}'] = round(ic, 4)
        row[f'pval_{t}'] = round(pval, 4)
    rows.append(row)

ic_df = pd.DataFrame(rows)
print(ic_df.to_string(index=False))

              factor  IC_fwd_ret_1d  pval_fwd_ret_1d  IC_fwd_ret_5d  pval_fwd_ret_5d  IC_fwd_ret_20d  pval_fwd_ret_20d
           vix_close         0.0509           0.0093         0.0887           0.0000          0.1661            0.0000
                 pcr         0.0443           0.0236         0.0251           0.2007          0.0615            0.0017
   max_pain_dist_pct        -0.0114           0.7960        -0.0176           0.6914         -0.1060            0.0181
               basis        -0.0817           0.0636        -0.1324           0.0027         -0.0729            0.1048
       cost_of_carry        -0.0301           0.5056        -0.0684           0.1317         -0.0065            0.8887
     fut_chng_oi_pct        -0.0256           0.1914        -0.0466           0.0172         -0.0024            0.9047
     fii_fut_net_pct         0.0289           0.1395         0.0259           0.1852          0.0352            0.0727
  client_fut_net_pct        -0.0318           0.

In [32]:
print(df[['vix_close','underlying_daily_vol','futures_daily_vol','applicable_daily_vol']].corr())

                      vix_close  underlying_daily_vol  futures_daily_vol  \
vix_close              1.000000              0.848139           0.850043   
underlying_daily_vol   0.848139              1.000000           0.999348   
futures_daily_vol      0.850043              0.999348           1.000000   
applicable_daily_vol   0.850593              0.999583           0.999873   

                      applicable_daily_vol  
vix_close                         0.850593  
underlying_daily_vol              0.999583  
futures_daily_vol                 0.999873  
applicable_daily_vol              1.000000  


In [33]:
# IC of max pain dist conditioned on being away from max pain
mask_tail = df['max_pain_dist_pct'].abs() > 1.5
sub = df[mask_tail & df['fwd_ret_20d'].notna() & df['max_pain_dist_pct'].notna()]
ic, pval = stats.spearmanr(sub['max_pain_dist_pct'], sub['fwd_ret_20d'])
print(f"Max pain tail IC (20D): {ic:.4f}, p={pval:.4f}, n={len(sub)}")

Max pain tail IC (20D): -0.4541, p=0.0070, n=34


In [34]:
TRAIN_END = '2025-06-20'
TEST_START = '2025-06-21'

# Verify the split
results = con.execute(f"""
    SELECT 
        CASE WHEN trade_date <= '{TRAIN_END}' THEN 'train' ELSE 'test' END AS split,
        COUNT(*) as days,
        MIN(trade_date) as from_date,
        MAX(trade_date) as to_date
    FROM daily_features
    WHERE fwd_ret_1d IS NOT NULL
    GROUP BY 1
    ORDER BY 1
""").fetchall()
for row in results: print(row)

('test', 268, datetime.date(2025, 6, 23), datetime.date(2026, 7, 22))
('train', 2346, datetime.date(2016, 1, 1), datetime.date(2025, 6, 20))


In [35]:
con.execute(f"""
    ALTER TABLE daily_features ADD COLUMN IF NOT EXISTS split VARCHAR;
    
    UPDATE daily_features 
    SET split = CASE 
        WHEN trade_date <= '{TRAIN_END}' THEN 'train' 
        ELSE 'test' 
    END
""")

In [36]:
df_train = con.execute("""
    SELECT vix_close, pcr, max_pain_dist_pct, basis, 
           cost_of_carry, fut_chng_oi_pct,
           fii_fut_net_pct, client_fut_net_pct,
           underlying_daily_vol, futures_daily_vol, applicable_daily_vol,
           adv_decl_ratio, traded_value_cr, num_trades, market_cap_cr, fii_net_flow_cr,
           fwd_ret_1d, fwd_ret_5d, fwd_ret_20d
    FROM daily_features
    WHERE fwd_ret_1d IS NOT NULL
      AND split = 'train'
""").df()

rows = []
for f in factors:
    row = {'factor': f}
    for t in targets:
        mask = df_train[f].notna() & df_train[t].notna()
        ic, pval = stats.spearmanr(df_train.loc[mask, f], df_train.loc[mask, t])
        row[f'IC_{t}'] = round(ic, 4)
        row[f'pval_{t}'] = round(pval, 4)
        row[f'n_{t}'] = mask.sum()
    rows.append(row)

ic_df_train = pd.DataFrame(rows)
print("=== IC Analysis — TRAIN ONLY (Jun 2024 - Jun 2025) ===")
print(ic_df_train.to_string(index=False))

=== IC Analysis — TRAIN ONLY (Jun 2024 - Jun 2025) ===
              factor  IC_fwd_ret_1d  pval_fwd_ret_1d  n_fwd_ret_1d  IC_fwd_ret_5d  pval_fwd_ret_5d  n_fwd_ret_5d  IC_fwd_ret_20d  pval_fwd_ret_20d  n_fwd_ret_20d
           vix_close         0.0448           0.0302          2346         0.0844           0.0000          2346          0.1475            0.0000           2346
                 pcr         0.0423           0.0405          2346         0.0084           0.6840          2346          0.0408            0.0480           2346
   max_pain_dist_pct         0.0433           0.4976           248        -0.0285           0.6556           248          0.0220            0.7305            248
               basis        -0.0893           0.1611           248        -0.2397           0.0001           248         -0.1130            0.0756            248
       cost_of_carry        -0.0588           0.3681           236        -0.1647           0.0113           236         -0.1072       

In [37]:
results = con.execute("""
    SELECT MIN(trade_date), MAX(trade_date), COUNT(*)
    FROM nse.fii_stats
    WHERE instrument = 'INDEX FUTURES'
""").fetchall()
print("fii_stats INDEX FUTURES coverage:", results)

results = con.execute("""
    SELECT strftime(trade_date, '%Y') AS yr, COUNT(*) AS days
    FROM nse.fii_stats
    WHERE instrument = 'INDEX FUTURES'
    GROUP BY 1 ORDER BY 1
""").fetchall()
print("\nBy year:")
for row in results: print(row)

fii_stats INDEX FUTURES coverage: [(datetime.date(2016, 1, 1), datetime.date(2026, 7, 23), 2615)]

By year:
('2016', 247)
('2017', 248)
('2018', 246)
('2019', 245)
('2020', 252)
('2021', 248)
('2022', 248)
('2023', 246)
('2024', 249)
('2025', 249)
('2026', 137)


In [38]:
results = con.execute("""
    SELECT
        NTILE(5) OVER (ORDER BY market_cap_cr) AS mcap_quintile,
        COUNT(*)                    AS days,
        ROUND(AVG(market_cap_cr), 0)   AS avg_mcap_cr,
        ROUND(AVG(fwd_ret_1d), 3)   AS avg_ret_1d,
        ROUND(AVG(fwd_ret_5d), 3)   AS avg_ret_5d,
        ROUND(AVG(fwd_ret_20d), 3)  AS avg_ret_20d,
        ROUND(AVG(up_1d) * 100, 1)  AS pct_up_days
    FROM daily_features
    WHERE fwd_ret_1d IS NOT NULL AND market_cap_cr IS NOT NULL
    GROUP BY 1
    ORDER BY 1
""").fetchall()

print("=== Market Cap Quintiles (full sample) ===")
for row in results: print(row)

results_train = con.execute("""
    SELECT
        NTILE(5) OVER (ORDER BY market_cap_cr) AS mcap_quintile,
        COUNT(*)                    AS days,
        ROUND(AVG(market_cap_cr), 0)   AS avg_mcap_cr,
        ROUND(AVG(fwd_ret_1d), 3)   AS avg_ret_1d,
        ROUND(AVG(fwd_ret_5d), 3)   AS avg_ret_5d,
        ROUND(AVG(fwd_ret_20d), 3)  AS avg_ret_20d,
        ROUND(AVG(up_1d) * 100, 1)  AS pct_up_days
    FROM daily_features
    WHERE fwd_ret_1d IS NOT NULL AND market_cap_cr IS NOT NULL AND split = 'train'
    GROUP BY 1
    ORDER BY 1
""").fetchall()

print("\n=== Market Cap Quintiles (TRAIN ONLY) ===")
for row in results_train: print(row)

BinderException: Binder Error: GROUP BY clause cannot contain window functions!

LINE 3:         NTILE(5) OVER (ORDER BY market_cap_cr) AS mcap_quintile,
                ^

In [ ]:
mask_tail = df_train['max_pain_dist_pct'].abs() > 1.5
sub = df_train[mask_tail & df_train['fwd_ret_20d'].notna() & df_train['max_pain_dist_pct'].notna()]
ic, pval = stats.spearmanr(sub['max_pain_dist_pct'], sub['fwd_ret_20d'])
print(f"\nMax pain tail IC 20D (train only): {ic:.4f}, p={pval:.4f}, n={len(sub)}")


Max pain tail IC 20D (train only): -0.1786, p=0.7017, n=7


In [ ]:
results = con.execute("""
    SELECT trade_date, max_pain_dist_pct, fwd_ret_20d, split, vix_close
    FROM daily_features
    WHERE ABS(max_pain_dist_pct) > 1.5
      AND fwd_ret_20d IS NOT NULL
    ORDER BY trade_date
""").fetchall()
for row in results: print(row)

(datetime.date(2024, 10, 7), -1.5476, -3.228, 'train', 15.08)
(datetime.date(2024, 10, 25), -1.6114, 0.0567, 'train', 14.6325)
(datetime.date(2024, 12, 20), -1.8952, -1.0291, 'train', 15.0725)
(datetime.date(2025, 1, 13), -1.5501, 2.0532, 'train', 15.9975)
(datetime.date(2025, 1, 27), -1.6451, -0.1456, 'train', 18.1325)
(datetime.date(2025, 4, 7), -2.4887, 8.3315, 'train', 22.7925)
(datetime.date(2025, 5, 12), 1.6132, 0.7162, 'train', 18.3925)
(datetime.date(2026, 3, 4), -2.5576, -6.1774, 'test', 21.14)
(datetime.date(2026, 3, 6), -1.7535, -1.8531, 'test', 19.88)
(datetime.date(2026, 3, 9), -1.6644, -1.0527, 'test', 23.3625)
(datetime.date(2026, 3, 11), -2.7409, -0.1014, 'test', 21.0625)
(datetime.date(2026, 3, 12), -3.0533, 2.505, 'test', 21.5175)
(datetime.date(2026, 3, 13), -3.3119, 4.5166, 'test', 22.645)
(datetime.date(2026, 3, 16), -1.8112, 4.0359, 'test', 21.6025)
(datetime.date(2026, 3, 18), -1.865, 3.3594, 'test', 18.7225)
(datetime.date(2026, 3, 19), -3.6389, 5.9818, 'test', 

In [ ]:
results = con.execute("""
    SELECT
        CASE
            WHEN vix_close < 14 THEN 'VIX_low'
            WHEN vix_close < 18 THEN 'VIX_mid'
            ELSE                     'VIX_high'
        END AS vix_regime,

        CASE
            WHEN basis < 0   THEN 'basis_discount'
            WHEN basis < 50  THEN 'basis_low'
            WHEN basis < 100 THEN 'basis_mid'
            ELSE                  'basis_high'
        END AS basis_regime,

        COUNT(*)                    AS days,
        ROUND(AVG(fwd_ret_1d), 3)   AS avg_ret_1d,
        ROUND(AVG(fwd_ret_5d), 3)   AS avg_ret_5d,
        ROUND(AVG(fwd_ret_20d), 3)  AS avg_ret_20d,
        ROUND(AVG(up_1d) * 100, 1)  AS pct_up_days

    FROM daily_features
    WHERE fwd_ret_1d IS NOT NULL
      AND split = 'train'
      AND vix_close IS NOT NULL
      AND basis IS NOT NULL

    GROUP BY 1, 2
    HAVING COUNT(*) >= 5
    ORDER BY 1, 2
""").fetchall()

print("=== VIX × Basis (TRAIN ONLY) ===")
for row in results: print(row)

=== VIX × Basis (TRAIN ONLY) ===
('VIX_high', 'basis_low', 7, -0.007, 0.579, 0.968, 42.9)
('VIX_high', 'basis_mid', 10, 0.928, 2.574, 4.512, 80.0)
('VIX_low', 'basis_discount', 10, 0.27, 0.817, -0.597, 60.0)
('VIX_low', 'basis_high', 16, -0.243, -1.225, 0.444, 31.3)
('VIX_low', 'basis_low', 27, 0.044, 0.133, 0.897, 63.0)
('VIX_low', 'basis_mid', 38, -0.1, -0.407, -0.458, 52.6)
('VIX_mid', 'basis_discount', 16, 0.343, 1.171, 1.401, 56.3)
('VIX_mid', 'basis_high', 21, 0.181, 0.211, -0.371, 57.1)
('VIX_mid', 'basis_low', 46, 0.067, 0.46, 0.948, 50.0)
('VIX_mid', 'basis_mid', 53, -0.199, -0.522, -0.517, 34.0)


In [ ]:
con.close()